In [100]:
import sympy as sp
import numpy as np
from scipy.linalg import expm
from numba import njit
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.ticker import LinearLocator
import random

In [82]:
# Funções de ativação:
@njit
def sigmoid(x):
    x = np.clip(x,-500,500)
    return 1/(1 + np.exp(-x))
@njit
def dsigmoid(x):
    x = np.clip(x,-500,500)
    return sigmoid(x)*(1-sigmoid(x))
@njit
def ReLU(x):
    x = np.clip(x,-500,500)
    return x * (x > 0)
@njit
def dReLU(x):
    x = np.clip(x,-500,500)
    return 1. * (x > 0)    
    
@njit
def tanhp(x):
    return np.tanh(x)
    
@njit
def dtanh(x):
    x = np.clip(x,-500,500)
    return 1-np.tanh(x)**2


# Função custo
@njit
def mse(y_pred, y_true):
    """
    Calcula o custo J (Erro Quadrático Médio).
    Retorna um único número (escalar).
    """
    return np.mean(np.square(y_pred - y_true))

def mse_derivative(y_pred, y_true):
    """
    Calcula a derivada de J em relação à previsão.
    Esse é o valor 'dA' que entra na ÚLTIMA camada para iniciar o backpropagation!
    """
    
    # A derivada de x^2 é 2x. Dividimos por 'm' para manter a média.
    return 2*(y_pred - y_true)

Utilizando a biblioteca numpy, podemos vetorizar as operações com uma única matriz de pesos (W) e um vetor de Viés (b).

$$
    \mathbf{Z} = \mathbf{W}\,\mathbf{x}. + \mathbf{b}
$$

A ideia será transformar isso numa operação em lote, ou seja, $x$ se torna uma matriz $\mathbf{X}$, e a operação se torna

$$
    \mathbf{Z} = \mathbf{W}\,\mathbf{X}. + \mathbf{b}
$$

Essa operação usando o numpy.dot é muito mais rápida, pois não irá utilizar laços for, tornando o cálculo muito mais eficiente, e agilizando o treinamento da rede neural.


* *Backpropagation* (training): considerando uma MLP com $N$ camadas.

* Para a camada de saída $N$ (*output layer*):

$$\delta^{(N-1)} = \mathbf{e} \odot g'(\mathbf{a}^{(N)}) \qquad \frac{\partial J}{\partial \mathbf{W}^{(N-1)}} = \delta^{(N-1)}(\mathbf{z}^{(N-1)})^T \qquad \frac{\partial J}{\partial \mathbf{b}^{(N)}} = \delta^{(N-1)}$$

* Para as camadas $n = N - 1$ até $2$ (*hidden layers + input layer*):

$$\delta^{(n-1)} = (\mathbf{W}^{(n)})^T \delta^{(n)} \odot f'(\mathbf{a}^{(n)}) \qquad \text{Quando } n = 2 \text{, temos } \mathbf{z}^{(1)} = \mathbf{x}.$$

$$\frac{\partial J}{\partial \mathbf{W}^{(n-1)}} = \delta^{(n-1)}(\mathbf{z}^{(n-1)})^T$$

$$\frac{\partial J}{\partial \mathbf{b}^{(n)}} = \d


Learning rate = tamanho do passo 
epocas = número de vezes que ele passa pelos mesmos dados.elta^{(n-1)}$$

In [110]:
# Definição de uma classe de Layer
class Layer:
    def __init__(self, 
                 num_inputs,                      # Numero de entradas
                 num_neurons,                     # Número de neurônios
                 activate_function = sigmoid,     # função de ativação
                 d_activate_function = dsigmoid): # Sua derivada.
        """ Quantas entradas ele terá e quantos neurônios terão"""
        # O shape da matriz será (entradas, neurônios)
        self.weights = 0.1 * np.random.randn(num_neurons, num_inputs) # neurons x inputs 
        self.biases = np.zeros((num_neurons,1))                       #vetor coluna 

        # Função de ativação escolhida
        self.output = None
        
        self._activate_function = activate_function      #função de ativação
        self._d_activate_function = d_activate_function  #derivada da função de ativação
        
        ## Variávies para backpropagation
        self.inputs = None 
        self.z = None 
        
    def forward(self, inputs):
        """ Operação para calcular a camada de forma vetorizada, podendo processar em lotes."""
        # Utilizado no backpropagation
        self.inputs = inputs 

        # z = W.x+ b
        self.z = np.dot(self.weights, inputs)+self.biases

        #Aplico a função de ativação 
        self.output = self._activate_function(self.z)
        
        return self.output

    def backward(self, dA, learning_rate):
        """
            dA: É o erro propagado da frente. 
            - Na última camada, equivale ao "e" da sua fórmula.
            - Nas camadas ocultas, equivale a (W^(n))^T * delta^(n).
        """
        # quantidade de exemplos no batch
        m = self.inputs.shape[1]

        # A atualização dos pesos de uma camada depende das entradas dessa camada
        # e do erro vindo da camada da frente
        # d^(n-1) = (W^(n)).T d^(n) * f'(a^(n))
        delta = dA * self._d_activate_function(self.z)

        # Atualizando os pesos
        dW = (1/m)*np.dot(delta, self.inputs.T)

        # Atualizando os biás e tornando vetor coluna
        db = (1/m) *np.sum(delta, axis=1, keepdims=True)

        # Erro que será passado para a camada anterior
        dA_prev = np.dot(self.weights.T, delta)

        # Aplicando taxa de aprendizado
        self.weights -= learning_rate *dW
        self.biases -= learning_rate *db

        return dA_prev
        
    def set_af(self,activate_function):
        self._activate_function = activate_function

    def set_daf(self, d_activate_function):
        self._d_activate_function = d_activate_function

# Definição da classe de uma rede neural
class NeuralNetwork:
    def __init__(self, layer_sizes, 
                 activate_function = sigmoid, 
                 d_activate_function= dsigmoid):
        """
        Repasso o formato da rede neural.
        Ex: [2,3,4,1] 
        2 - input_layey
        3 e 4 -> layers escondidos
        1 - saída 
        """
        self.layers = []
        for i in range(1,len(layer_sizes)):
            # Gerando o layer
            # Camada anterior
            num_inputs  =  layer_sizes[i-1]
            # Camada atual 
            num_neurons = layer_sizes[i]

            # Adiciona a camada e inicializa ela
            self.layers.append(Layer(num_inputs, num_neurons,activate_function, d_activate_function))

    def forward(self, inputs):
        # vou passando a diante os inputs
        current_input = np.array(inputs) 

        if current_input.ndim == 1:
            current_input = current_input.reshape(-1, 1)

        expected_inputs = self.layers[0].weights.shape[1]
        if current_input.shape[0] != expected_inputs and current_input.shape[1] == expected_inputs:
            # Transpõe a matriz. Agora as linhas são as features e as colunas são os exemplos.
            current_input = current_input.T
            
        for layer in self.layers:
            current_input = layer.forward(current_input)


        return current_input

    def train(self, 
              inputs, 
              targets, 
              epochs=1000,
              learning_rate=0.1):
        """
        inputs: Dados de entrada (features x exemplos) ou (exemplos, features) – será ajustado.
        targets: Resultados esperados (saídas x exemplos) ou (exemplos, saídas) – será ajustado.
        epochs: Quantas vezes a rede verá todos os dados.
        """
        X,Y = self.adjust(inputs, targets) #Garantir que X e Y sejam np.arrays do tipo coluna e devidamente ajustados para lotes.
        m = X.shape[1]  # quantidade de exemplos no batch
    
        # Loop de treinamento
        for epoch in range(epochs):
            # Forward
            predictions = self.forward(X)
    
            # Cálculo da perda (apenas para exibição)
            if epoch % 100 == 0 or epoch == epochs - 1:
                loss = mse(predictions, Y)
                #print(f"Época {epoch} | Erro (MSE): {loss:.4f}")
    
            # Erro propagável (dA) – assume mse_derivative = predictions - Y
            dA = mse_derivative(predictions, Y)   # shape (saídas, m)
    
            # Backward por todas as camadas (ordem inversa)
            for layer in reversed(self.layers):
                dA = layer.backward(dA, learning_rate)

    
    def adjust(self, X, Y):
        X = np.array(X)
        Y = np.array(Y)
    
        # -- Ajuste seguro dos inputs (X) --
        if X.ndim == 1:
            X = X.reshape(-1, 1)
        
        expected_inputs = self.layers[0].weights.shape[1]   # número de features esperado
        if X.shape[0] != expected_inputs:                   # se linhas não são as features
            if X.shape[1] == expected_inputs:               # colunas são as features? então transpõe
                X = X.T
            else:
                raise ValueError(
                    f"Shape de X inválido: esperado (features={expected_inputs}, exemplos) "
                    f"ou (exemplos, features={expected_inputs}), mas recebido {X.shape}"
                )

        # -- Ajuste seguro dos targets (Y) --
        if Y.ndim == 1:
            Y = Y.reshape(-1, 1)
        
        expected_outputs = self.layers[-1].weights.shape[0] # número de saídas da rede
        if Y.shape[0] != expected_outputs:                  # se linhas não são as saídas
            if Y.shape[1] == expected_outputs:              # colunas são as saídas? então transpõe
                Y = Y.T
            else:
                raise ValueError(
                    f"Shape de Y inválido: esperado (saídas={expected_outputs}, exemplos) "
                    f"ou (exemplos, saídas={expected_outputs}), mas recebido {Y.shape}"
                )
        # agora Y tem shape (saídas, m)
    
        # -- Verifica consistência do número de exemplos --
        if X.shape[1] != Y.shape[1]:
            raise ValueError(
                f"Número de exemplos incompatível: X tem {X.shape[1]}, Y tem {Y.shape[1]}"
            )

        return X,Y

In [108]:
# Exemplo para verificar se está funcionando
nn = NeuralNetwork([2,4,1],tanhp, dtanh)

entradas = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
]).T

previsoes = nn.forward(entradas)

print("Previsões (ainda aleatórias, pois a rede não foi treinada):")
print(previsoes)

Previsões (ainda aleatórias, pois a rede não foi treinada):
[[ 0.         -0.00386544 -0.02101099 -0.02491189]]


In [114]:
# Rede: 2 Entradas, 4 Neurônios Ocultos, 1 Saída
nn = NeuralNetwork([2, 5, 4, 1],tanhp, dtanh)

# Entradas (Batch de 4 exemplos)
X = [
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
]

# Respostas Esperadas (XOR)
Y = [
    [0],
    [1],
    [1],
    [0]
]

print("Treinando a rede...")
nn.train(X, Y, epochs=1000, learning_rate=0.3)

print("\nResultados Finais Pós-Treinamento:")
previsoes = nn.forward(X).T # .T apenas para printar bonitinho na tela
print(np.round(previsoes, 3))

Treinando a rede...

Resultados Finais Pós-Treinamento:
[[0.493]
 [0.499]
 [0.501]
 [0.506]]
